# Machine Learning Financial Analysis

This notebook analyzes financial data, classifies metrics into Pros (values > 10%) and Cons (values < 10%), generates human-readable statements, selects the top 3 pros and cons per company, and returns results as a structured Python dictionary.

In [5]:
import pandas as pd
import numpy as np
import json
from datetime import datetime

In [6]:
def _safe_float(x):
    try:
        return float(x)
    except:
        return np.nan

def parse_percent_string(s):
    """
    Parse strings like '3 Years: 35%' or '5 Years: 19%' or '35%' or '35' .
    Returns (value_float_or_nan, period_str_or_empty)
    """
    if pd.isna(s):
        return (np.nan, '')
    if isinstance(s, (int, float)):
        return (float(s), '')
    s = str(s).strip()
    # common formats: "3 Years: 35%", "5 Years : 19%", "35%", "35"
    # try to split on ':' first
    if ':' in s:
        left, right = s.split(':', 1)
        period = left.strip()
        val_str = right.strip()
    else:
        period = ''
        val_str = s
    # remove percent and commas, parentheses
    val_str = val_str.replace('%', '').replace(',', '').replace('(', '-').replace(')','').strip()
    try:
        val = float(val_str)
        return (val, period)
    except:
        return (np.nan, period)

def score_metric(value, threshold=10.0, direction='higher_is_better'):
    """
    Returns a score reflecting how strongly a metric is pro/cons.
    If direction='higher_is_better', then:
      - value > threshold => positive score = value - threshold
      - value < threshold => negative score = threshold - value
    Score is absolute distance; sign indicates pro (>0) or con (<0) relative to threshold.
    """
    if np.isnan(value):
        return 0.0
    diff = value - threshold
    if direction == 'higher_is_better':
        return diff  # positive => pro, negative => con
    else:
        # lower is better (e.g., leverage): invert sign so negative diff means pro
        return -diff

def generate_context_statements_from_bs_pl_cf(bs_df=None, pl_df=None, cf_df=None):
    """
    From latest balance-sheet / profitloss / cashflow data frames (if provided),
    compute simple context-based statements per company_id.
    Returns dict: {company_id: {'pros': [(text,score)], 'cons': [(text,score)]}}
    """
    out = {}
    # helper to get latest row per company by year_dt if available
    def latest_rows(df):
        if df is None or df.empty:
            return pd.DataFrame()
        if 'year_dt' in df.columns:
            return df.sort_values(['company_id','year_dt']).groupby('company_id').tail(1).set_index('company_id')
        else:
            return df.sort_values('id').groupby('company_id').tail(1).set_index('company_id')
    bs_latest = latest_rows(bs_df)
    pl_latest = latest_rows(pl_df)
    cf_latest = latest_rows(cf_df)

    company_ids = set()
    for d in (bs_latest, pl_latest, cf_latest):
        if not d.empty:
            company_ids.update(d.index.tolist())

    for cid in company_ids:
        pros, cons = [], []
        # Leverage / debt ratio
        if cid in bs_latest.index:
            r = bs_latest.loc[cid]
            borrowings = _safe_float(r.get('borrowings', np.nan))
            total_assets = _safe_float(r.get('total_assets', np.nan))
            if not np.isnan(borrowings) and not np.isnan(total_assets) and total_assets != 0:
                lev = borrowings / total_assets
                # thresholds: lev < 0.05 => almost debt-free, <0.2 => low leverage
                if lev < 0.05:
                    text = f"Company is almost debt-free (leverage {lev:.2%})."
                    pros.append((text, 1.0 + (0.05 - lev)))  # small boost for being very low
                elif lev < 0.20:
                    text = f"Low leverage: borrowings are {lev:.2%} of assets."
                    pros.append((text, 0.5 + (0.20 - lev)))
                elif lev > 0.5:
                    text = f"High leverage: borrowings are {lev:.2%} of assets."
                    cons.append((text, lev - 0.5))
        # Interest coverage
        if cid in pl_latest.index:
            r = pl_latest.loc[cid]
            op = _safe_float(r.get('operating_profit', np.nan))
            interest = _safe_float(r.get('interest', np.nan))
            if not np.isnan(op) and not np.isnan(interest) and interest != 0:
                ic = op / interest
                if ic > 8:
                    pros.append((f"Very comfortable interest coverage (OP/Interest = {ic:.1f}).", ic-8))
                elif ic > 3:
                    pros.append((f"Good interest coverage (OP/Interest = {ic:.1f}).", ic-3))
                elif ic < 1:
                    cons.append((f"Interest burden is high (OP/Interest = {ic:.2f}).", 1-ic))
        # Cashflow quality
        if cid in cf_latest.index:
            r = cf_latest.loc[cid]
            net_cf = _safe_float(r.get('net_cash_flow', np.nan))
            if not np.isnan(net_cf):
                if net_cf > 0:
                    pros.append((f"Positive net cash flow in latest period: {net_cf:.2f}.", net_cf / (abs(net_cf) + 1)))
                else:
                    cons.append((f"Negative net cash flow in latest period: {net_cf:.2f}.", -net_cf / (abs(net_cf) + 1)))
        out[cid] = {'pros': pros, 'cons': cons}
    return out

def classify_and_generate_statements(
    analysis_csv,
    meta_csv,
    bs_csv=None,
    pl_csv=None,
    cf_csv=None,
    threshold=10.0,
    top_k=3,
    save_json_path=None,
    save_csv_path=None
):
    """
    Main function.
    - analysis_csv: path to analysis_master.csv
    - meta_csv: path to companies_meta.csv
    - optional bs_csv/pl_csv/cf_csv to generate extra context-aware statements
    - threshold: percent threshold for pros/cons (default 10%)
    - top_k: how many top pros/cons to return per company
    """
    analysis_df = pd.read_csv(analysis_csv)
    meta_df = pd.read_csv(meta_csv) if meta_csv else pd.DataFrame()

    # optional supplemental tables (for context-aware rules)
    bs_df = pd.read_csv(bs_csv) if bs_csv else pd.DataFrame()
    pl_df = pd.read_csv(pl_csv) if pl_csv else pd.DataFrame()
    cf_df = pd.read_csv(cf_csv) if cf_csv else pd.DataFrame()

    # Precompute supplemental context-based statements
    context_statements = generate_context_statements_from_bs_pl_cf(
        bs_df=bs_df if not bs_df.empty else None,
        pl_df=pl_df if not pl_df.empty else None,
        cf_df=cf_df if not cf_df.empty else None
    )

    # metric templates (can be extended)
    metric_templates = {
        'compounded_sales_growth': {
            'pro': "Strong sales growth: {val:.1f}% over {period}.",
            'con': "Weak sales growth: only {val:.1f}% over {period}."
        },
        'compounded_profit_growth': {
            'pro': "Profit growth is robust: {val:.1f}% over {period}.",
            'con': "Profit growth is low: {val:.1f}% over {period}."
        },
        'roe': {
            'pro': "High return on equity: {val:.1f}% ({period}).",
            'con': "Low return on equity: {val:.1f}% ({period})."
        },
        # numeric fields from analysis/meta
        'roce_percentage': {
            'pro': "Strong return on capital employed: {val:.1f}%.",
            'con': "Low return on capital employed: {val:.1f}%."
        },
        'roe_percentage': {
            'pro': "Healthy average ROE: {val:.1f}%.",
            'con': "ROE is low: {val:.1f}%."
        }
    }

    results = []
    # Group by company_id. analysis_df may have multiple rows per company (3y/5y/10y)
    for company_id, group in analysis_df.groupby('company_id'):
        pros_candidates = []
        cons_candidates = []

        # row-level metrics (compounded_sales_growth, compounded_profit_growth, roe)
        for idx, row in group.iterrows():
            # Compounded Sales Growth
            sales_raw = row.get('compounded_sales_growth', None)
            sales_val, sales_period = parse_percent_string(sales_raw)
            if not np.isnan(sales_val):
                sc = score_metric(sales_val, threshold=threshold, direction='higher_is_better')
                if sc > 0:
                    txt = metric_templates['compounded_sales_growth']['pro'].format(val=sales_val, period=sales_period)
                    pros_candidates.append((txt, sc))
                else:
                    txt = metric_templates['compounded_sales_growth']['con'].format(val=sales_val, period=sales_period)
                    cons_candidates.append((txt, -sc))

            # Compounded Profit Growth
            profit_raw = row.get('compounded_profit_growth', None)
            profit_val, profit_period = parse_percent_string(profit_raw)
            if not np.isnan(profit_val):
                sc = score_metric(profit_val, threshold=threshold, direction='higher_is_better')
                if sc > 0:
                    txt = metric_templates['compounded_profit_growth']['pro'].format(val=profit_val, period=profit_period)
                    pros_candidates.append((txt, sc))
                else:
                    txt = metric_templates['compounded_profit_growth']['con'].format(val=profit_val, period=profit_period)
                    cons_candidates.append((txt, -sc))

            # ROE (may be stored under 'roe' or as 'roe' like "3 Years: 7%")
            roe_raw = row.get('roe', None)
            roe_val, roe_period = parse_percent_string(roe_raw)
            if not np.isnan(roe_val):
                sc = score_metric(roe_val, threshold=threshold, direction='higher_is_better')
                if sc > 0:
                    txt = metric_templates['roe']['pro'].format(val=roe_val, period=roe_period)
                    pros_candidates.append((txt, sc))
                else:
                    txt = metric_templates['roe']['con'].format(val=roe_val, period=roe_period)
                    cons_candidates.append((txt, -sc))

        # analysis-level numeric fields (roce_percentage, roe_percentage) that may be present in the analysis row(s)
        for col in ['roce_percentage', 'roe_percentage']:
            if col in group.columns:
                # may have multiple rows; pick max non-null (or mean)
                vals = pd.to_numeric(group[col], errors='coerce').dropna()
                if not vals.empty:
                    val = float(vals.iloc[0])  # take first/representative - you can change to mean()
                    sc = score_metric(val, threshold=threshold, direction='higher_is_better')
                    if sc > 0:
                        txt = metric_templates[col]['pro'].format(val=val)
                        pros_candidates.append((txt, sc))
                    else:
                        txt = metric_templates[col]['con'].format(val=val)
                        cons_candidates.append((txt, -sc))

        # include context statements (from balance sheet / profit & loss / cashflow)
        ctx = context_statements.get(company_id, {})
        for t, s in ctx.get('pros', []):
            pros_candidates.append((t, s))
        for t, s in ctx.get('cons', []):
            cons_candidates.append((t, s))

        # Deduplicate similar text candidates (keep highest score)
        def dedup_keep_best(candidates):
            best_map = {}
            for txt, sc in candidates:
                k = txt.strip()
                if k in best_map:
                    best_map[k] = max(best_map[k], sc)
                else:
                    best_map[k] = sc
            # return sorted list by score desc
            return sorted([(k, v) for k, v in best_map.items()], key=lambda x: -x[1])

        pros_sorted = dedup_keep_best(pros_candidates)
        cons_sorted = dedup_keep_best(cons_candidates)

        # select top_k
        pros_top = [{'text': t, 'score': float(s)} for t, s in pros_sorted[:top_k]]
        cons_top = [{'text': t, 'score': float(s)} for t, s in cons_sorted[:top_k]]

        # company name lookup
        company_name = company_id
        if not meta_df.empty and 'company_id' in meta_df.columns:
            rowm = meta_df[meta_df['company_id'] == company_id]
            if not rowm.empty and 'company_name' in rowm.columns:
                company_name = rowm.iloc[0]['company_name']

        results.append({
            'company_id': company_id,
            'company_name': company_name,
            'pros': pros_top,
            'cons': cons_top,
            'last_updated': datetime.utcnow().isoformat()  # UTC timestamp
        })

    # Optionally save results to JSON/CSV
    if save_json_path:
        with open(save_json_path, 'w', encoding='utf-8') as fh:
            json.dump(results, fh, indent=2, ensure_ascii=False)
    if save_csv_path:
        # simple flat CSV with pros/cons concatenated
        rows = []
        for r in results:
            rows.append({
                'company_id': r['company_id'],
                'company_name': r['company_name'],
                'pros': ' ||| '.join([p['text'] for p in r['pros']]),
                'cons': ' ||| '.join([c['text'] for c in r['cons']]),
                'last_updated': r['last_updated']
            })
        pd.DataFrame(rows).to_csv(save_csv_path, index=False)

    return results


# ---------------------------
# Example usage (adjust paths)
# ---------------------------
if __name__ == '__main__':
    results = classify_and_generate_statements(
        analysis_csv='../Task_2/compiled_output/analysis_master.csv',
        meta_csv='../Task_2/compiled_output/companies_meta.csv',
        bs_csv='../Task_2/compiled_output/balancesheet_master.csv' if False else None,  # add if available
        pl_csv='../Task_2/compiled_output/profitloss_master.csv' if False else None,    # add if available
        cf_csv='../Task_2/compiled_output/cashflow_master.csv' if False else None,      # add if available
        threshold=10.0,
        top_k=3,
        save_json_path='ml_analysis_results.json',
        save_csv_path='ml_analysis_results.csv'
    )

    # print first result
    import pprint
    pprint.pprint(results[0])

{'company_id': 'ADANIENSOL',
 'company_name': 'Adani Energy Solutions Ltd',
 'cons': [{'score': 12.0, 'text': 'Profit growth is low: -2.0% over 3 Years.'},
          {'score': 1.4100000000000001, 'text': 'ROE is low: 8.6%.'},
          {'score': 1.0, 'text': 'Low return on capital employed: 9.0%.'}],
 'last_updated': '2025-08-15T12:09:30.439310',
 'pros': [{'score': 9.0, 'text': 'Strong sales growth: 19.0% over 3 Years.'},
          {'score': 8.0, 'text': 'Strong sales growth: 18.0% over 5 Years.'},
          {'score': 6.0,
           'text': 'Profit growth is robust: 16.0% over 5 Years.'}]}


C:\Users\Chethan\AppData\Local\Temp\ipykernel_9984\80600105.py:275: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'last_updated': datetime.utcnow().isoformat()  # UTC timestamp
